# Scope 3.1 Carbon Data Catalog
**Portfolioprojekt | Datenmanagement | Juni 2026**

Prototypischer Datenkatalog zur systematischen Identifikation, Dokumentation und
Lückenerkennung von Scope-3.1-Emissionsdaten auf Basis des GHG Protocol.
Vollständige Projektbeschreibung: `01_kontext.md`

## Semantische Beziehungskette

Requirement → Business Term → Technical Field → Dataset → Source System
                                                    ↓
                                               Data Owner
                                                    ↓
                                            Metadata Gap

Eine Anforderung gilt als vollständig abgedeckt, wenn die gesamte Kette
lückenlos dokumentiert und einem Owner zugewiesen ist.

## Aufbau dieses Notebooks

| Abschnitt | Inhalt |
|---|---|
| **1** | Datenbankschema & Initialisierung |
| **2** | Seed-Daten: 8 Scope-3.1-Anforderungen |
| **3** | Gap Detection Engine |
| **4** | Interaktive Abfrage & Curation |
| **5** | Coverage-Visualisierung |
| **6** | Fazit & Ausblick |

In [1]:
#Umgebungscheck
import sqlite3
import pandas as pd

try:
    import plotly.express as px
    print("✓ plotly verfügbar")
except ImportError:
    print("→ pip install plotly")

try:
    import ipywidgets as widgets
    print("✓ ipywidgets verfügbar")
except ImportError:
    print("→ pip install ipywidgets")

print("✓ sqlite3 und pandas bereit")

→ pip install plotly
→ pip install ipywidgets
✓ sqlite3 und pandas bereit


In [2]:
import sqlite3
import pandas as pd

# Datenbankverbindung herstellen
conn = sqlite3.connect("catalog.db")
cursor = conn.cursor()

# Fremdschlüssel aktivieren
cursor.execute("PRAGMA foreign_keys = ON")

print("✓ catalog.db erstellt und verbunden")

✓ catalog.db erstellt und verbunden


In [ ]:
import sqlite3
import pandas as pd

# ── Datenbankverbindung ──────────────────────────────────────────
conn = sqlite3.connect("catalog.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

# ── Tabellen in korrekter Reihenfolge erstellen ──────────────────
# Reihenfolge ist kritisch: referenzierte Tabelle muss zuerst existieren

# 1. Quellsysteme
cursor.execute("""
CREATE TABLE IF NOT EXISTS source_systems (
    id                      INTEGER PRIMARY KEY AUTOINCREMENT,
    name                    TEXT NOT NULL,
    system_type             TEXT NOT NULL,
    subsidiary              TEXT NOT NULL,
    country                 TEXT NOT NULL,
    currency                TEXT NOT NULL,
    is_central              INTEGER NOT NULL DEFAULT 0,
    consolidation_status    TEXT NOT NULL,
    consolidation_target    TEXT,
    data_availability       TEXT NOT NULL,
    owner                   TEXT,
    notes                   TEXT
)
""")

# 2. Datensätze
cursor.execute("""
CREATE TABLE IF NOT EXISTS datasets (
    id                      INTEGER PRIMARY KEY AUTOINCREMENT,
    name                    TEXT NOT NULL,
    description             TEXT,
    source_system_id        INTEGER NOT NULL,
    data_contact            TEXT,
    data_contact_role       TEXT,
    data_contact_reliability TEXT,
    currency                TEXT,
    exchange_rate_date      TEXT,
    data_availability       TEXT NOT NULL,
    data_entry_type         TEXT DEFAULT 'automatisch',
    evidence_available      TEXT DEFAULT 'unbekannt',
    evidence_type           TEXT,
    manual_entry_count      INTEGER DEFAULT 0,
    FOREIGN KEY (source_system_id) REFERENCES source_systems(id)
)
""")

# 3. Technische Felder
cursor.execute("""
CREATE TABLE IF NOT EXISTS technical_fields (
    id                              INTEGER PRIMARY KEY AUTOINCREMENT,
    dataset_id                      INTEGER NOT NULL,
    field_name                      TEXT NOT NULL,
    field_description               TEXT,
    data_type                       TEXT,
    unit                            TEXT,
    unit_system                     TEXT,
    material_description_quality    TEXT,
    field_owner                     TEXT,
    is_nullable                     INTEGER DEFAULT 1,
    FOREIGN KEY (dataset_id) REFERENCES datasets(id)
)
""")

# 4. Business Terms
cursor.execute("""
CREATE TABLE IF NOT EXISTS business_terms (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    name            TEXT NOT NULL,
    definition      TEXT,
    owner           TEXT,
    domain          TEXT,
    tier_level      INTEGER
)
""")

# 5. Anforderungen
cursor.execute("""
CREATE TABLE IF NOT EXISTS requirements (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    name                TEXT NOT NULL,
    description         TEXT NOT NULL,
    source              TEXT NOT NULL,
    tier_level          INTEGER NOT NULL,
    priority            TEXT NOT NULL,
    business_term_id    INTEGER,
    technical_field_id  INTEGER,
    coverage_status     TEXT DEFAULT 'offen',
    FOREIGN KEY (business_term_id)  REFERENCES business_terms(id),
    FOREIGN KEY (technical_field_id) REFERENCES technical_fields(id)
)
""")

# 6. Metadaten-Lücken (wird durch Gap Detection befüllt)
cursor.execute("""
CREATE TABLE IF NOT EXISTS metadata_gaps (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    gap_type            TEXT NOT NULL,
    gap_subtype         TEXT DEFAULT NULL,
    severity            TEXT NOT NULL,
    affected_entity     TEXT NOT NULL,
    affected_id         INTEGER NOT NULL,
    description         TEXT NOT NULL,
    recommended_action  TEXT,
    status              TEXT DEFAULT 'offen',
    created_at          TEXT DEFAULT (datetime('now'))
)
""")

conn.commit()
print("✓ Schema erfolgreich erstellt — 6 Tabellen angelegt")
print("✓ Fremdschlüssel aktiv")
print("✓ catalog.db bereit für Seed-Daten")

In [4]:
# Prüfzelle: Tabellen anzeigen
tables = pd.read_sql_query("""
    SELECT name FROM sqlite_master 
    WHERE type='table' 
    ORDER BY name
""", conn)

print("Angelegte Tabellen:")
print(tables.to_string(index=False))

Angelegte Tabellen:
            name
  business_terms
        datasets
   metadata_gaps
    requirements
  source_systems
 sqlite_sequence
technical_fields


In [18]:
# Prüfzelle: Überblick Seed-Daten
for table in ["source_systems", "datasets", "technical_fields", 
              "business_terms", "requirements"]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) as anzahl FROM {table}", conn
    )
    print(f"{table:25s} → {count['anzahl'][0]} Einträge")

source_systems            → 9 Einträge
datasets                  → 10 Einträge
technical_fields          → 18 Einträge
business_terms            → 10 Einträge
requirements              → 11 Einträge


In [16]:
# Datenbank vollständig zurücksetzen
# Reihenfolge wichtig: zuerst abhängige Tabellen löschen

cursor.execute("DROP TABLE IF EXISTS metadata_gaps")
cursor.execute("DROP TABLE IF EXISTS requirements")
cursor.execute("DROP TABLE IF EXISTS business_terms")
cursor.execute("DROP TABLE IF EXISTS technical_fields")
cursor.execute("DROP TABLE IF EXISTS datasets")
cursor.execute("DROP TABLE IF EXISTS source_systems")
cursor.execute("DELETE FROM sqlite_sequence")

conn.commit()
print("✓ Alle Tabellen gelöscht")
print("✓ Datenbank bereit für neues Schema")

✓ Alle Tabellen gelöscht
✓ Datenbank bereit für neues Schema


In [19]:
for table in ["source_systems", "datasets", "technical_fields",
              "business_terms", "requirements"]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) as anzahl FROM {table}", conn
    )
    print(f"{table:25s} → {count['anzahl'][0]} Einträge")

source_systems            → 9 Einträge
datasets                  → 10 Einträge
technical_fields          → 18 Einträge
business_terms            → 10 Einträge
requirements              → 11 Einträge


In [20]:
import pandas as pd

# Alle Tabellen anzeigen
tabellen = ["source_systems", "datasets", "technical_fields", 
            "business_terms", "requirements"]

for tabelle in tabellen:
    print(f"\n{'='*60}")
    print(f"  {tabelle.upper()}")
    print(f"{'='*60}")
    df = pd.read_sql_query(f"SELECT * FROM {tabelle}", conn)
    display(df)


  SOURCE_SYSTEMS


,id,name,system_type,subsidiary,country,currency,is_central,consolidation_status,consolidation_target,data_availability,owner,notes
0,1,SAP S/4HANA + BW,Konsolidierung,FloorTec EU,Deutschland,EUR,1,vollständig,SAP BW EU,vollständig,Anna Müller,None
1,2,SAP FI,Buchhaltung,FloorTec DE,Deutschland,EUR,0,vollständig,SAP BW EU,vollständig,Thomas Becker,None
2,3,SAP FI,Buchhaltung,FloorTec FR,Frankreich,EUR,0,vollständig,SAP BW EU,partiell,Claire Dubois,None
3,4,Salesforce + QuickBooks,CRM,FloorTec USA,USA,USD,0,nicht_angeschlossen,NaN,partiell,NaN,None
4,5,SAP S/4HANA,ERP,FloorTec MY,Malaysia,MYR,0,nicht_angeschlossen,NaN,partiell,NaN,None
5,6,Oracle Fusion,ERP,FloorTec CN,China,CNY,0,nicht_angeschlossen,NaN,unbekannt,NaN,None
6,7,Microsoft D365,ERP,FloorTec AU,Australien,AUD,0,nicht_angeschlossen,NaN,partiell,NaN,None
7,8,Kreditkartenabrechnung,informell,FloorTec MY,Malaysia,MYR,0,nicht_angeschlossen,NaN,unbekannt,NaN,None
8,9,Manuelle Excel,informell,FloorTec CN,China,CNY,0,nicht_angeschlossen,NaN,unbekannt,NaN,None



  DATASETS


,id,name,description,source_system_id,data_contact,data_contact_role,data_contact_reliability,currency,exchange_rate_date,data_availability,data_entry_type,evidence_available,evidence_type,manual_entry_count
0,1,MM_PURCHASE_ORDERS_EU,Zentrale Einkaufsbestellungen EU,1,Anna Müller,Controlling,hoch,EUR,2024-12-31,vollständig,automatisch,True,Bestellung,0
1,2,FI_VENDOR_SPEND_DE,Lieferantenausgaben Deutschland,2,Thomas Becker,Controlling,hoch,EUR,2024-12-31,vollständig,automatisch,True,Rechnung,0
2,3,FI_VENDOR_SPEND_FR,Lieferantenausgaben Frankreich,3,Claire Dubois,Controlling,mittel,EUR,2024-12-31,partiell,semi-manuell,True,Rechnung,12
3,4,QB_VENDOR_SPEND_USA,Lieferantenausgaben USA (QuickBooks),4,NaN,unbekannt,niedrig,USD,NaN,partiell,manuell,False,NaN,47
4,5,SF_PURCHASE_ORDERS_USA,Einkaufsbestellungen USA (Salesforce),4,NaN,unbekannt,niedrig,USD,NaN,partiell,manuell,False,NaN,31
5,6,SAP_GOODS_RECEIPT_MY,Wareneingänge Malaysia (Rohstoffe),5,NaN,GL-Assistenz,niedrig,MYR,NaN,partiell,semi-manuell,unbekannt,Lieferschein,28
6,7,ORA_PURCHASE_ORDERS_CN,Einkaufsbestellungen China,6,NaN,GL-Assistenz,niedrig,CNY,NaN,unbekannt,manuell,False,NaN,64
7,8,D365_GOODS_RECEIPT_AU,Wareneingänge Australien (Mineralien),7,NaN,IT,mittel,AUD,NaN,partiell,semi-manuell,True,Lieferschein,15
8,9,SHADOW_CC_MY,Kreditkartenabrechnungen Malaysia,8,NaN,GL-Assistenz,niedrig,MYR,NaN,unbekannt,manuell,False,NaN,83
9,10,SHADOW_XLS_CN,Manuelle Excel-Exporte China,9,NaN,GL-Assistenz,niedrig,CNY,NaN,unbekannt,manuell,False,NaN,97



  TECHNICAL_FIELDS


,id,dataset_id,field_name,field_description,data_type,unit,unit_system,material_description_quality,field_owner,is_nullable
0,1,1,LIFNR,Lieferantennummer,TEXT,ID,metrisch,lesbar,Anna Müller,0
1,2,1,NETWR,Nettowert Einkauf EUR,NUMERIC,EUR,metrisch,lesbar,Anna Müller,0
2,3,1,MATKL,Materialgruppe,TEXT,NaN,metrisch,lesbar,Thomas Becker,0
3,4,2,LIFNR,Lieferantennummer DE,TEXT,ID,metrisch,lesbar,Thomas Becker,0
4,5,2,NETWR,Nettowert EUR,NUMERIC,EUR,metrisch,lesbar,Thomas Becker,0
5,6,3,LIFNR,Lieferantennummer FR,TEXT,ID,metrisch,lesbar,Claire Dubois,0
6,7,3,NETWR,Nettowert EUR FR,NUMERIC,EUR,metrisch,teilweise_lesbar,NaN,1
7,8,4,VendorID,Lieferanten-ID USA,TEXT,ID,metrisch,lesbar,NaN,0
8,9,4,Amount_USD,Ausgabenbetrag USD,NUMERIC,USD,metrisch,lesbar,NaN,0
9,10,5,MatCode,Materialcode Salesforce,TEXT,NaN,metrisch,teilweise_lesbar,NaN,1



  BUSINESS_TERMS


,id,name,definition,owner,domain,tier_level
0,1,Lieferant,Externes Unternehmen das Waren oder Dienstleis...,Anna Müller,Einkauf,2
1,2,Materialgruppe,Klassifikation von Materialien nach Typ und Ve...,Thomas Becker,Einkauf,2
2,3,Einkaufswert,Nettobetrag einer Einkaufsbestellung in Buchun...,Anna Müller,Finanzen,3
3,4,Emissionsfaktor,kg CO2e pro Einheit Material oder Ausgabe (EXI...,NaN,Nachhaltigkeit,2
4,5,Materialherkunftsland,Land der Rohstoffgewinnung oder Produktion,NaN,Nachhaltigkeit,2
5,6,Einheit,"Mengeneinheit einer Einkaufsposition (kg, lbs,...",NaN,Einkauf,2
6,7,Währung,Buchungswährung je Gesellschaft,Anna Müller,Finanzen,3
7,8,Wechselkurs,Jahresdurchschnittskurs zur EUR-Konversion,NaN,Finanzen,3
8,9,Verifikationsstatus,Nachweis ob Lieferantendaten extern geprüft wu...,NaN,Nachhaltigkeit,1
9,10,Lieferantenklassifikation,Einstufung nach Emissionsrelevanz und Klimarisiko,NaN,Nachhaltigkeit,3



  REQUIREMENTS


,id,name,description,source,tier_level,priority,business_term_id,technical_field_id,coverage_status
0,1,Lieferantenspez. THG-Daten,Produktspezifische Emissionen direkt vom Liefe...,GHG Protocol,1,low,9.0,NaN,offen
1,2,Verifikationsstatus,Nachweis externer Prüfung von Lieferantendaten,ESRS E1,1,low,9.0,NaN,offen
2,3,Eingekaufte Menge je Lieferant,Menge je Lieferant und Material für massebasie...,GHG Protocol,2,high,6.0,11.0,partiell
3,4,Einheitennormierung,Konvertierung lokaler und imperialer Einheiten...,GHG Protocol,2,high,6.0,NaN,offen
4,5,Materialgruppe harmonisiert,Lesbare Materialgruppe als Basis für EF-Mapping,GHG Protocol,2,high,2.0,12.0,partiell
5,6,Emissionsfaktor je Matgruppe,EXIOBASE-Faktor in kg CO2e pro Einheit,GHG Protocol,2,high,4.0,NaN,offen
6,7,Materialherkunftsland,Herkunftsland für EF-Splitting und Klimarisiko,ESRS E1,2,medium,5.0,NaN,offen
7,8,Einkaufswert je Lieferant,Einkaufswert in EUR nach Währungskonversion,GHG Protocol,3,medium,3.0,2.0,partiell
8,9,Quellsystem je Gesellschaft,Identifikation des führenden Systems je Tochte...,GHG Protocol,3,medium,NaN,NaN,partiell
9,10,Lieferantenklassifikation,"Einstufung nach Emissionsrelevanz, Länderrisik...",ESRS E1,3,medium,10.0,NaN,offen


In [21]:
# Data Dictionary aller Tabellen
dd_query = """
    SELECT 
        m.name        AS Tabelle,
        p.name        AS Feldname,
        p.type        AS Datentyp,
        CASE p."notnull" 
            WHEN 1 THEN 'Pflichtfeld' 
            ELSE 'Optional' 
        END           AS Pflicht,
        CASE p.pk 
            WHEN 1 THEN 'Primärschlüssel' 
            ELSE '' 
        END           AS Schlüssel,
        p.dflt_value  AS Standardwert
    FROM sqlite_master m
    JOIN pragma_table_info(m.name) p
    WHERE m.type = 'table'
    AND m.name != 'sqlite_sequence'
    ORDER BY m.name, p.cid
"""

dd = pd.read_sql_query(dd_query, conn)

for tabelle in dd["Tabelle"].unique():
    print(f"\n{'='*60}")
    print(f"  TABELLE: {tabelle.upper()}")
    print(f"{'='*60}")
    display(dd[dd["Tabelle"] == tabelle][
        ["Feldname","Datentyp","Pflicht","Schlüssel","Standardwert"]
    ].reset_index(drop=True))


  TABELLE: BUSINESS_TERMS


,Feldname,Datentyp,Pflicht,Schlüssel,Standardwert
0,id,INTEGER,Optional,Primärschlüssel,NaN
1,name,TEXT,Pflichtfeld,,NaN
2,definition,TEXT,Optional,,NaN
3,owner,TEXT,Optional,,NaN
4,domain,TEXT,Optional,,NaN
5,tier_level,INTEGER,Optional,,NaN



  TABELLE: DATASETS


,Feldname,Datentyp,Pflicht,Schlüssel,Standardwert
0,id,INTEGER,Optional,Primärschlüssel,NaN
1,name,TEXT,Pflichtfeld,,NaN
2,description,TEXT,Optional,,NaN
3,source_system_id,INTEGER,Pflichtfeld,,NaN
4,data_contact,TEXT,Optional,,NaN
5,data_contact_role,TEXT,Optional,,NaN
6,data_contact_reliability,TEXT,Optional,,NaN
7,currency,TEXT,Optional,,NaN
8,exchange_rate_date,TEXT,Optional,,NaN
9,data_availability,TEXT,Pflichtfeld,,NaN



  TABELLE: METADATA_GAPS


...

,Feldname,Datentyp,Pflicht,Schlüssel,Standardwert
0,id,INTEGER,Optional,Primärschlüssel,NaN
1,name,TEXT,Pflichtfeld,,NaN
2,description,TEXT,Pflichtfeld,,NaN
3,source,TEXT,Pflichtfeld,,NaN
4,tier_level,INTEGER,Pflichtfeld,,NaN
5,priority,TEXT,Pflichtfeld,,NaN
6,business_term_id,INTEGER,Optional,,NaN
7,technical_field_id,INTEGER,Optional,,NaN
8,coverage_status,TEXT,Optional,,'offen'



  TABELLE: SOURCE_SYSTEMS


,Feldname,Datentyp,Pflicht,Schlüssel,Standardwert
0,id,INTEGER,Optional,Primärschlüssel,NaN
1,name,TEXT,Pflichtfeld,,NaN
2,system_type,TEXT,Pflichtfeld,,NaN
3,subsidiary,TEXT,Pflichtfeld,,NaN
4,country,TEXT,Pflichtfeld,,NaN
5,currency,TEXT,Pflichtfeld,,NaN
6,is_central,INTEGER,Pflichtfeld,,0
7,consolidation_status,TEXT,Pflichtfeld,,NaN
8,consolidation_target,TEXT,Optional,,NaN
9,data_availability,TEXT,Pflichtfeld,,NaN



  TABELLE: TECHNICAL_FIELDS


,Feldname,Datentyp,Pflicht,Schlüssel,Standardwert
0,id,INTEGER,Optional,Primärschlüssel,NaN
1,dataset_id,INTEGER,Pflichtfeld,,NaN
2,field_name,TEXT,Pflichtfeld,,NaN
3,field_description,TEXT,Optional,,NaN
4,data_type,TEXT,Optional,,NaN
5,unit,TEXT,Optional,,NaN
6,unit_system,TEXT,Optional,,NaN
7,material_description_quality,TEXT,Optional,,NaN
8,field_owner,TEXT,Optional,,NaN
9,is_nullable,INTEGER,Optional,,1


In [22]:
# Fremdschlüsselbeziehungen anzeigen
tabellen = ["datasets", "technical_fields", "requirements"]

print("FREMDSCHLÜSSELBEZIEHUNGEN\n")
for tabelle in tabellen:
    fk = pd.read_sql_query(
        f"PRAGMA foreign_key_list({tabelle})", conn
    )
    if not fk.empty:
        print(f"{tabelle}")
        for _, row in fk.iterrows():
            print(f"  └── {row['from']:25s} → {row['table']}.{row['to']}")
        print()

FREMDSCHLÜSSELBEZIEHUNGEN

datasets
  └── source_system_id          → source_systems.id

technical_fields
  └── dataset_id                → datasets.id

requirements
  └── technical_field_id        → technical_fields.id
  └── business_term_id          → business_terms.id



In [23]:
import sys
sys.path.append("src")

from gap_detector import GapDetector

# GapDetector initialisieren
detector = GapDetector("catalog.db")

✓ GapDetector verbunden mit catalog.db


In [1]:
import sqlite3
import pandas as pd

In [2]:
import sys
sys.path.append("src")
from gap_detector import GapDetector

detector = GapDetector("catalog.db")

✓ GapDetector verbunden mit catalog.db


In [3]:
detector.gaps_leeren()
ergebnis_r1 = detector.check_fehlender_field_owner()

print(f"\nBetroffene Felder:")
print(ergebnis_r1[["field_name", "dataset_name", 
                    "subsidiary", "is_nullable"]].to_string(index=False))

✓ Bestehende Gaps geleert
✓ Regel 1 ausgeführt — 11 Gaps gefunden

Betroffene Felder:
field_name           dataset_name   subsidiary  is_nullable
     NETWR     FI_VENDOR_SPEND_FR  FloorTec FR            1
  VendorID    QB_VENDOR_SPEND_USA FloorTec USA            0
Amount_USD    QB_VENDOR_SPEND_USA FloorTec USA            0
   MatCode SF_PURCHASE_ORDERS_USA FloorTec USA            1
     MENGE   SAP_GOODS_RECEIPT_MY  FloorTec MY            0
     MATKL   SAP_GOODS_RECEIPT_MY  FloorTec MY            1
       QTY ORA_PURCHASE_ORDERS_CN  FloorTec CN            1
  MAT_DESC ORA_PURCHASE_ORDERS_CN  FloorTec CN            1
  MATERIAL  D365_GOODS_RECEIPT_AU  FloorTec AU            1
    Betrag           SHADOW_CC_MY  FloorTec MY            1
   Ausgabe          SHADOW_XLS_CN  FloorTec CN            1


In [ ]:
# Regel 2 testen: Quellsysteme ohne Konsolidierungsanbindung
detector.gaps_leeren()
ergebnis_r2 = detector.check_systemanschluss()

print(f"\nNicht angeschlossene Systeme:")
print(ergebnis_r2[["name", "subsidiary", "country", "consolidation_status"]].to_string(index=False))

In [ ]:
# Verifizierung: Gaps in metadata_gaps Tabelle prüfen
gaps_check = pd.read_sql_query("""
    SELECT gap_type, severity, affected_entity, description
    FROM metadata_gaps
    WHERE gap_type = 'kein_systemanschluss'
    ORDER BY affected_id
""", detector.conn)

print(f"Einträge in metadata_gaps: {len(gaps_check)} (erwartet: 6)")
display(gaps_check)

In [ ]:
# run_all(): alle Regeln in einem Durchlauf
detector.run_all()

In [ ]:
# Coverage Score: Anteil vollständig abgedeckter Anforderungen
ergebnis_coverage = detector.coverage_score()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
sys.path.append("src")
from query_engine import QueryEngine

qe = QueryEngine("catalog.db")

# ── Widgets ──────────────────────────────────────────────────────
dd_severity = widgets.Dropdown(
    options=["alle", "high", "medium", "low"],
    description="Severity:",
    style={"description_width": "80px"}
)
dd_gap_type = widgets.Dropdown(
    options=qe.gap_typen(),
    description="Gap-Typ:",
    style={"description_width": "80px"}
)
dd_entity = widgets.Dropdown(
    options=qe.entitaeten(),
    description="Entität:",
    style={"description_width": "80px"}
)
btn_filtern = widgets.Button(
    description="Filtern",
    button_style="primary"
)
output = widgets.Output()

# ── Button-Aktion ─────────────────────────────────────────────────
def on_filtern(b):
    with output:
        clear_output(wait=True)
        df = qe.gaps_filtern(
            severity=dd_severity.value,
            gap_type=dd_gap_type.value,
            entity=dd_entity.value
        )
        print(f"{len(df)} Gap(s) gefunden\n")
        display(df[["gap_type","gap_subtype","severity",
                     "affected_entity","description","status"]])

btn_filtern.on_click(on_filtern)

# ── Layout ────────────────────────────────────────────────────────
display(
    widgets.VBox([
        widgets.HTML("<b>Gap-Browser</b>"),
        widgets.HBox([dd_severity, dd_gap_type, dd_entity, btn_filtern]),
        output
    ])
)

In [ ]:
# ── Anforderungs-Browser nach Tier ───────────────────────────────
dd_tier = widgets.Dropdown(
    options=["alle", "1", "2", "3"],
    description="Tier:",
    style={"description_width": "80px"}
)
btn_tier = widgets.Button(description="Filtern", button_style="primary")
output_tier = widgets.Output()

def on_tier(b):
    with output_tier:
        clear_output(wait=True)
        tier = None if dd_tier.value == "alle" else int(dd_tier.value)
        df = qe.requirements_nach_tier(tier=tier)
        print(f"{len(df)} Anforderung(en)\n")
        display(df)

btn_tier.on_click(on_tier)

display(
    widgets.VBox([
        widgets.HTML("<b>Anforderungs-Browser (Coverage nach Tier)</b>"),
        widgets.HBox([dd_tier, btn_tier]),
        output_tier
    ])
)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# ── Chart 1: Coverage Score Donut ────────────────────────────────
df_cov = qe.coverage_daten()

farben_cov = {
    "vollständig": "#2ecc71",
    "partiell":    "#f39c12",
    "offen":       "#e74c3c"
}

fig_donut = px.pie(
    df_cov,
    names="coverage",
    values="anzahl",
    hole=0.55,
    color="coverage",
    color_discrete_map=farben_cov,
    title="Coverage Score — Scope 3.1 Anforderungen (FloorTec Group)"
)
fig_donut.update_traces(textinfo="label+percent", textfont_size=13)
fig_donut.update_layout(
    annotations=[dict(
        text="9.1%<br>vollständig",
        x=0.5, y=0.5,
        font_size=16,
        showarrow=False
    )],
    legend_title="Status",
    height=420
)
fig_donut.show()

In [ ]:
# ── Chart 2: Gaps nach Typ und Severity ──────────────────────────
df_gaps = qe.gap_uebersicht()

farben_sev = {
    "high":   "#e74c3c",
    "medium": "#f39c12",
    "low":    "#3498db"
}

fig_bar = px.bar(
    df_gaps,
    x="anzahl",
    y="gap_type",
    color="severity",
    color_discrete_map=farben_sev,
    orientation="h",
    barmode="stack",
    title="Metadatenlücken nach Typ und Severity (FloorTec Group)",
    labels={
        "anzahl":   "Anzahl Gaps",
        "gap_type": "Gap-Typ",
        "severity": "Severity"
    },
    category_orders={"severity": ["high", "medium", "low"]}
)
fig_bar.update_layout(
    height=400,
    yaxis=dict(autorange="reversed"),
    legend_title="Severity"
)
fig_bar.show()

In [ ]:
# Regel 3 testen: Unleserliche / fehlende Materialbeschreibungen
detector.gaps_leeren()
ergebnis_r3 = detector.check_unleserliche_materialbeschreibung()

print(f"\nBetroffene Felder:")
print(ergebnis_r3[["field_name", "dataset_name", "subsidiary",
                    "material_description_quality"]].to_string(index=False))